<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Jour 3 (bis) — Distiller vers un CNN plutôt que vers des k-mers
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Réutiliser le teacher du Jour 2 pour entraîner un second student, un CNN one-hot<br>
    - Comparer les deux students : le CNN dépasse-t-il le plafond de la représentation k-mer ?<br>
    - Mettre en regard précision, nombre de paramètres et latence<br>
</div>

<div style="border-left: 4px solid #0969da; background-color: #f2f6fd; padding: 0.9em 1.1em; color: #24292f;">
  <strong style="color: #0969da;">Version corrigée &mdash; solution de référence</strong><br>
  Ce notebook contient <strong>toutes les réponses</strong>. Il n'est pas à compléter et n'est pas à rendre.<br>
  Il prolonge <code>03_knowledge_distillation.ipynb</code> : exécutez-le après lui.
</div>

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

Le student du notebook 03 ne voit que des **fréquences de k-mers** : l'ordre des
nucléotides et le contexte long ont déjà été jetés. Quelle que soit la qualité de la
distillation, il ne peut pas dépasser ce que cette représentation contient.

Ici on garde **exactement le même teacher et les mêmes cibles douces**, mais on donne au
student une entrée plus riche : la séquence brute encodée en one-hot, lue par un CNN.
Seule la représentation change --- c'est la variable qu'on teste.

In [1]:
import sys
sys.path.append("src")

from pathlib import Path
import numpy as np
import pandas as pd
import torch

from data import load_all
from embeddings import load_supervised_embeddings
from featurize import kmer_matrix, one_hot_batch
from models.classifier_heads import MLPHead
from models.baselines import OneHotCNN
from models.distillation import train_student
from eval import evaluate_logits, count_params, measure_latency_torch

EMB_DIR = "../../2-data/embeddings"
PROCESSED_DIR = "../../2-data/processed"
TEACHER_PATH = Path("../../2-data/models/teacher_mlp.pt")
WINDOW = 200

In [2]:
X_train_emb, y_train, ids_train = load_supervised_embeddings(EMB_DIR, "train")
X_val_emb, y_val, ids_val = load_supervised_embeddings(EMB_DIR, "val")

# le teacher entraîné au notebook 02 (gelé : il ne sert qu'à produire des cibles douces)
ckpt = torch.load(TEACHER_PATH)
teacher = MLPHead(d_in=ckpt["d_in"]); teacher.load_state_dict(ckpt["state_dict"]); teacher.eval()

# les séquences des MÊMES fenêtres, retrouvées par leur id
splits = load_all(PROCESSED_DIR, max_rows=None)
train_df, val_df = splits["train"].set_index("id"), splits["val"].set_index("id")
train_seqs = train_df.loc[ids_train, "sequence"].tolist()
val_seqs = val_df.loc[ids_val, "sequence"].tolist()

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

with torch.no_grad():
    teacher_logits_train = teacher(torch.tensor(X_train_emb, dtype=torch.float32))
    teacher_val_logits = teacher(torch.tensor(X_val_emb, dtype=torch.float32)).numpy()

teacher_metrics = evaluate_logits(teacher_val_logits, y_val)
print("teacher :", teacher_metrics)
print(len(train_seqs), "fenêtres d'entraînement |", len(val_seqs), "de validation")

teacher : {'accuracy': 0.959, 'f1': 0.9595261599210266}
4000 fenêtres d'entraînement | 1000 de validation


<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Le student de référence : k-mers + MLP**

In [3]:
# --- student de référence : k-mers + petit MLP (celui du notebook 03)
X_train_kmer = torch.tensor(kmer_matrix(train_seqs, k=4), dtype=torch.float32)
X_val_kmer = torch.tensor(kmer_matrix(val_seqs, k=4), dtype=torch.float32)

torch.manual_seed(0)
student_kmer = MLPHead(d_in=X_train_kmer.shape[1], d_hidden=32)
student_kmer, _ = train_student(student_kmer, teacher_logits_train, X_train_kmer, y_train_t,
                                 epochs=150, temperature=4.0, alpha=0.5, log_every=50)

student_kmer.eval()
with torch.no_grad():
    kmer_metrics = evaluate_logits(student_kmer(X_val_kmer).numpy(), y_val)
print("student k-mer :", kmer_metrics)

epoch   1/150 | loss=5.8882 | acc=0.5000
epoch  50/150 | loss=4.5038 | acc=0.8007
epoch 100/150 | loss=3.6885 | acc=0.8592
epoch 150/150 | loss=3.3851 | acc=0.8718
student k-mer : {'accuracy': 0.838, 'f1': 0.8396039603960396}


#### **Le nouveau student : one-hot + CNN**

`teacher_logits_train` ne change pas : le teacher note des *fenêtres*, pas des représentations.

In [4]:
# --- nouveau student : CNN one-hot sur la séquence brute
X_train_oh = torch.tensor(one_hot_batch(train_seqs, length=WINDOW))
X_val_oh = torch.tensor(one_hot_batch(val_seqs, length=WINDOW))
print("forme des entrées CNN :", tuple(X_train_oh.shape), "(batch, 4 bases, longueur)")

torch.manual_seed(0)
student_cnn = OneHotCNN(seq_len=WINDOW)
student_cnn, history_cnn = train_student(
    student_cnn, teacher_logits_train, X_train_oh, y_train_t,
    epochs=150, temperature=4.0, alpha=0.5, log_every=25,
)

student_cnn.eval()
with torch.no_grad():
    cnn_metrics = evaluate_logits(student_cnn(X_val_oh).numpy(), y_val)
print("student CNN :", cnn_metrics)

forme des entrées CNN : (4000, 4, 200) (batch, 4 bases, longueur)
epoch   1/150 | loss=5.8809 | acc=0.5128
epoch  25/150 | loss=4.4693 | acc=0.7768
epoch  50/150 | loss=3.8815 | acc=0.8277
epoch  75/150 | loss=3.7085 | acc=0.8397
epoch 100/150 | loss=3.6132 | acc=0.8482
epoch 125/150 | loss=3.6553 | acc=0.8452
epoch 150/150 | loss=3.4952 | acc=0.8568
student CNN : {'accuracy': 0.807, 'f1': 0.8230980751604033}


#### **Comparaison**

In [5]:
rows = [
    {"modèle": "teacher (Evo2 + MLP)", **teacher_metrics,
     "params": count_params(teacher),
     "latence_ms": measure_latency_torch(teacher,
                       torch.tensor(X_val_emb[:1], dtype=torch.float32))},
    {"modèle": "student k-mer", **kmer_metrics,
     "params": count_params(student_kmer),
     "latence_ms": measure_latency_torch(student_kmer, X_val_kmer[:1])},
    {"modèle": "student CNN", **cnn_metrics,
     "params": count_params(student_cnn),
     "latence_ms": measure_latency_torch(student_cnn, X_val_oh[:1])},
]
comparison = pd.DataFrame(rows)

# part de la performance du teacher récupérée par chaque student
base = teacher_metrics["accuracy"]
comparison["% du teacher"] = (comparison["accuracy"] / base * 100).round(1)
comparison

,modèle,accuracy,f1,params,latence_ms,% du teacher
0,teacher (Evo2 + MLP),0.959,0.959526,524545,0.051477,100.0
1,student k-mer,0.838,0.839604,8257,0.022576,87.4
2,student CNN,0.807,0.823098,10465,0.132598,84.2


#### **Point de contrôle**

Le résultat n'est pas celui qu'on attendait, et c'est intéressant : avec ce budget de
données, le **student CNN fait moins bien que le student k-mer** (0,808 contre 0,838),
tout en étant plus lourd et environ dix fois plus lent à l'inférence.

Regardez les traces d'entraînement : le CNN atteint 0,859 en apprentissage pour 0,808 en
validation. L'écart dit tout — il **surapprend**. Un CNN doit découvrir ses motifs à
partir de zéro, ce qui demande beaucoup d'exemples ; les fréquences de k-mers, elles,
sont un *a priori* déjà très informatif, offert gratuitement. Dans le régime « peu de
données étiquetées » (4000 fenêtres), l'a priori gagne.

Ce qu'il faudrait pour renverser le résultat :

- **plus de fenêtres** — c'est le levier principal, le CNN passe devant quand les données
  abondent ;
- de la **régularisation** (dropout, augmentation de données par complément inverse) ;
- éventuellement plus de `channels`, mais sans plus de données cela aggraverait plutôt
  le surapprentissage.

La leçon à retenir, et elle vaut au-delà de cet atelier : **une représentation plus
expressive n'est pas automatiquement meilleure**. Elle l'est quand on a de quoi
l'entraîner. C'est exactement pour cette raison qu'on utilise un modèle de fondation
comme Evo2 : il a déjà payé ce coût d'apprentissage sur d'énormes quantités d'ADN, et
nous en récupérons le bénéfice avec seulement quelques milliers d'exemples étiquetés.

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14s16 7 16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Jour 3 &mdash; piste CNN</div>
      <div>Suite &rarr; <code>03c_ablation_temperature_alpha.ipynb</code></div>
    </div>
    <div style="flex: 0 0 auto; text-align: right; border-right: 2px solid #0969da; padding-right: 0.9em;">
      <div style="font-weight: 600; color: #24292f;">EEIA &middot; bioAI Workshop</div>
      <div style="font-size: 0.85em;">Semaine 4 &mdash; De l'ADN aux mod&egrave;les compress&eacute;s</div>
    </div>
  </div>
</div>